# 01. Leíró Statisztika és Feltáró Adatelemzés (EDA)

**Cél:** az eladó lakáspiac fundamentális ár-, méret- és elrendezési struktúrájának statisztikai vizsgálata.

---

### 📖 A nyers átlagárak torzító hatása
A lakáspiaci elemzések gyakori hibája a számtani átlag kizárólagos használata. A piac extrém módon heterogén: a standard garzonoktól az egyedi árazású luxusingatlanokig terjed, az eloszlás felső szegmense hosszú „farkat" képez, amely az átlagot felfelé torzítja. A fejezet az árak valódi eloszlását, a robusztus kvartiliseket és a logaritmikus transzformáció indokoltságát mutatja be.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from ingatlan_tdk._utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm

df = load_szamitott_master()
display(HTML(render_area_header_html(df=df)))
elado = df[df['listing_type'] == 'elado'].copy()
AREA_NAME = get_area_metadata()['name']
print(f"Eladó lakások száma az EDA mintában: {len(elado)} db.")


### 1. Középértékek és Szóródási Mutatók

**📌 Az eredmények értelmezése:**
- **Átlag vs. medián:** a kettő közötti eltérés pontosan számszerűsíti a felső árszegmens torzító hatását; a piac valódi árszintjét a medián négyzetméterár reprezentálja.
- **Relatív szórás:** a magas (20% feletti) relatív szórás arra utal, hogy az árakat dominánsan a lokáció és az épületminőség diverzitása magyarázza, nem csupán az alapterület.

**🔍 A kvartilisek jelentősége:** Q1 a „belépő" árszint, Q3 a prémium szegmens határa, az IQR (interkvartilis terjedelem) a piac középső 50%-ának szélsőérték-mentes sávszélessége.

In [ ]:
skew_nm = stats.skew(elado['nm_ar_huf'].dropna())
skew_dir = "Jobbra ferde" if skew_nm > 0 else ("Balra ferde" if skew_nm < 0 else "Szimmetrikus")

kpi_cards = [
    ("Átlagos Kínálati Ár", fmt_mft(elado['price_huf'].mean() / 1e6), f"Medián: {fmt_mft(elado['price_huf'].median() / 1e6)}", "#1e3a8a"),
    ("Átlagos Ár / m²", fmt_huf(elado['nm_ar_huf'].mean()), f"Medián: {fmt_huf(elado['nm_ar_huf'].median())}", "#2563eb"),
    ("Átlagos Alapterület", f"{elado['alapterulet_nm'].mean():.1f} m²", f"Medián: {elado['alapterulet_nm'].median():.1f} m²", "#059669"),
    ("Átlagos Szobaszám", f"{elado['szobaszam_osszes'].mean():.1f} szoba", f"Módusz: {elado['szobaszam_osszes'].mode()[0]} szoba", "#d97706"),
    ("Ár / m² Szórás", fmt_huf(elado['nm_ar_huf'].std()), f"Relatív szórás: {elado['nm_ar_huf'].std()/elado['nm_ar_huf'].mean()*100:.1f}%", "#dc2626"),
    ("Ár / m² Ferdeség", f"{skew_nm:.2f}", f"{skew_dir} eloszlás", "#7c3aed")
]
display(HTML(kpi_grid_html(kpi_cards)))

# Részletes leíró táblázat
desc_vars = ['price_huf', 'nm_ar_huf', 'alapterulet_nm', 'szobaszam_osszes']
desc_names = ['Ár (HUF)', 'Ár / m² (HUF)', 'Alapterület (m²)', 'Szobaszám']
stat_df = elado[desc_vars].describe().T
stat_df['skew'] = [stats.skew(elado[c].dropna()) for c in desc_vars]
stat_df['kurtosis'] = [stats.kurtosis(elado[c].dropna()) for c in desc_vars]
stat_df.index = desc_names
stat_df.columns = ['Darab', 'Átlag', 'Szórás', 'Min', '25% (Q1)', 'Medián (Q2)', '75% (Q3)', 'Max', 'Ferdeség', 'Csúcsosság']

html_table = "<div style='overflow-x:auto; margin: 15px 0;'>" + stat_df.round(2).to_html(classes='table table-bordered table-hover') + "</div>"
display(HTML(html_table))


### 2. Áreloszlás és Logaritmikus Transzformáció

**📌 A hisztogramok összehasonlítása:**
- **Bal ábra:** a nyers fajlagos árak eloszlása — tipikusan jobbra ferde.
- **Jobb ábra:** a természetes logaritmus (ln(ár/m²)) alkalmazásával kapott eloszlás — közelíti a normálist.
- **Formális ellenőrzés:** a „közelítőleg normális" állítást az alábbi kódcella **Jarque–Bera** és **Shapiro–Wilk** normalitástesztekkel támasztja alá (mindkét teszt nullhipotézise a normalitás).

**🔍 Ökonometriai indoklás:**
1. A lineáris regresszió (OLS) alapfeltétele a hibatagok normál eloszlása; a nyers árak aszimmetriája sérti ezt, a log-transzformáció „kisimítja".
2. A logaritmikus függő változó mellett a becsült együtthatók közvetlenül **százalékos prémiumként/diszkontként** (marginal willingness to pay) értelmezhetők.

In [ ]:
fig1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Nyers Négyzetméterár Eloszlása (Ft/m²)', 'Logaritmikus Négyzetméterár Eloszlása (ln(Ár/m²))')
)

fig1.add_trace(
    go.Histogram(x=elado['nm_ar_huf'], nbinsx=35, name='Ár/m²', marker_color='#2563eb', opacity=0.8),
    row=1, col=1
)

fig1.add_trace(
    go.Histogram(x=np.log(elado['nm_ar_huf']), nbinsx=35, name='ln(Ár/m²)', marker_color='#059669', opacity=0.8),
    row=1, col=2
)

fig1.update_layout(
    title_text='Fajlagos árak eloszlása és normálissá alakítása log-transzformációval',
    template=PLOTLY_TEMPLATE,
    height=450,
    showlegend=False
)
fig1.show()

# Formális normalitásvizsgálat: a "közelítőleg normális" állítás statisztikai alátámasztása
# (Jarque–Bera és Shapiro–Wilk; mindkét teszt nullhipotézise a normalitás)
print("Normalitásvizsgálat az eladó mintán (H0: normális eloszlás):")
for col, nev in [('nm_ar_huf', 'nm_ar_huf (nyers fajlagos ár)'), ('log_nm_ar', 'log_nm_ar (log-transzformált)')]:
    s = elado[col].dropna()
    jb = stats.jarque_bera(s)
    sh = stats.shapiro(s)
    jb_ok = "H0 megtartva" if jb.pvalue >= 0.05 else "H0 elutasítva"
    sh_ok = "H0 megtartva" if sh.pvalue >= 0.05 else "H0 elutasítva"
    print(
        f"  {nev}: n = {len(s)} | Jarque–Bera: stat = {jb.statistic:.1f}, p = {jb.pvalue:.2e} ({jb_ok}) | "
        f"Shapiro–Wilk: stat = {sh.statistic:.4f}, p = {sh.pvalue:.2e} ({sh_ok})"
    )


### 3. Dobozdiagramok (Boxplot) és Kiugró Értékek Városrészenként

**📌 Hogyan kell olvasni a dobozdiagramot?**
- **A doboz közepe (vastag vonal)**: A medián ár (a középső lakás értéke).
- **A doboz alsó és felső éle**: Az alsó kvartilis (Q1) és felső kvartilis (Q3). A doboz magassága maga a piac középső 50%-a (IQR).
- **A függőleges bajszok**: A normális ártartomány határai ($1.5 \times \text{IQR}$).
- **A bajszokon kívüli egyedi pontok**: Statisztikai kiugró értékek (outlierek).

**🔍 Mit látunk Kőbánya városrészeiben?**
- **Óhegy és Ligettelek**: A dobozok magasabban helyezkednek el, a felső kvartilis eléri az 1.3 - 1.4 millió Ft/m²-es szintet.
- **Újhegy**: A doboz rendkívül tömör és feszes! A lakások szabványos méretei és paneles jellege miatt az árak alacsony szórásúak, a medián ~1.1 millió Ft/m² körül stabil.
- **Gyárdűlő és Laposdűlő**: Hosszabb dobozok és nagyobb szóródás, mivel a vegyes ipari-rozsdaövezeti környezetben az elhanyagolt épületek és a modern új fejlesztések árai erősen eltérnek egymástól.

**💡 Tanulság**: Újhegy a legkiszámíthatóbb árú övezet, Óhegy a legváltozatosabb prémium zóna.

In [ ]:
# Városrészi boxplot
fig2 = px.box(
    elado,
    x='varosresz',
    y='nm_ar_huf',
    color='varosresz',
    title='Négyzetméterár dobozábra (Boxplot) városrészenként',
    labels={'varosresz': 'Városrész', 'nm_ar_huf': 'Ár / m² (HUF)'},
    template=PLOTLY_TEMPLATE
)
fig2.update_layout(xaxis_tickangle=-30, height=450, showlegend=False)
fig2.show()

# Méret vs Ár szórásdiagram OLS trenddel — a becslés együtthatója, p-értéke és R²-e a címben
ols_df = elado[['alapterulet_nm', 'ar_millio_ft']].dropna()
X_ols = sm.add_constant(ols_df['alapterulet_nm'])
ols_res = sm.OLS(ols_df['ar_millio_ft'], X_ols).fit()
ols_b = ols_res.params['alapterulet_nm']
ols_p = ols_res.pvalues['alapterulet_nm']
ols_r2 = ols_res.rsquared

fig3 = px.scatter(
    elado,
    x='alapterulet_nm',
    y='ar_millio_ft',
    color='varosresz',
    trendline='ols',
    title=(
        f'Alapterület vs. Kínálati Ár összefüggés — OLS: együttható = {ols_b:.3f} M Ft/m², '
        f'p = {ols_p:.2e}, R² = {ols_r2:.3f} (n = {len(ols_df)})'
    ),
    labels={'alapterulet_nm': 'Alapterület (m²)', 'ar_millio_ft': 'Ár (Millió Ft)', 'varosresz': 'Városrész'},
    template=PLOTLY_TEMPLATE
)
fig3.update_layout(height=480)
fig3.show()


### 4. Korrelációs Hőtérkép és Változókapcsolatok

**📌 Mit mutat a korrelációs hőtérkép?**
Az alábbi ábra a kulcsváltozók **Pearson-** és **Spearman-**korrelációs mátrixát ábrázolja két panelben, az eladó mintára (listwise teljes megfigyelésekkel):
- **Pozitív érték (kék):** a változók együtt mozognak.
- **Negatív érték (piros):** ellentétes irányú együttmozgás (pl. a nagyobb lakások fajlagos m² ára a garzon-effektus miatt enyhén alacsonyabb).
- **Szignifikancia-csillagok:** a cellák csillagozása a páronkénti korreláció p-értékét jelöli (* p < 0.05, ** p < 0.01, *** p < 0.001).
- **Spearman vs. Pearson:** a rangkorreláció a monoton (nem feltétlenül lineáris) kapcsolatokat is kimutatja, ezért robusztusabb a kiugró értékekre.

**💡 Fő tanulság**: A méret, az állapot és a közlekedési elérhetőség (vasúti távolság, POI-sűrűség) a piaci ár legfőbb strukturális mozgatórugói — ezeket a további fejezetek formális modellekkel vizsgálják.


In [ ]:
# Tényleges korrelációs hőtérkép: Pearson + Spearman mátrix, szignifikancia-csillagokkal (p < 0.05)
corr_vars = [
    'nm_ar_huf', 'alapterulet_nm', 'szobaszam_osszes', 'epulet_kora_ev',
    'allapot_kod', 'tavolsag_vasut_m', 'tavolsag_vasut_halozati_m', 'poi_15p_count'
]
corr_labels = {
    'nm_ar_huf': 'Ár / m²',
    'alapterulet_nm': 'Alapterület',
    'szobaszam_osszes': 'Szobaszám',
    'epulet_kora_ev': 'Épület kora',
    'allapot_kod': 'Állapot kód',
    'tavolsag_vasut_m': 'Vasút távolság',
    'tavolsag_vasut_halozati_m': 'Vasút hálózati táv.',
    'poi_15p_count': 'POI-k 15 perc',
}
corr_df = elado[corr_vars].dropna()  # listwise teljes minta

def corr_pvalues(df, method):
    """Páronkénti korrelációs p-érték mátrix (Pearson vagy Spearman)."""
    cols = list(df.columns)
    pmat = pd.DataFrame(np.eye(len(cols)), index=cols, columns=cols)
    fn = stats.pearsonr if method == 'pearson' else stats.spearmanr
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            res = fn(df.iloc[:, i], df.iloc[:, j])
            p = res.pvalue if hasattr(res, 'pvalue') else res[1]
            pmat.iloc[i, j] = p
            pmat.iloc[j, i] = p
    return pmat

def star_cells(rho, pmat):
    """Korrelációs értékek csillagozott szöveges mátrixa (* p<0.05, ** p<0.01, *** p<0.001)."""
    txt = pd.DataFrame('', index=rho.index, columns=rho.columns)
    for i in rho.index:
        for j in rho.columns:
            p = pmat.loc[i, j]
            if p < 0.001:
                stars = '***'
            elif p < 0.01:
                stars = '**'
            elif p < 0.05:
                stars = '*'
            else:
                stars = ''
            txt.loc[i, j] = f"{rho.loc[i, j]:.2f}{stars}"
    return txt

pearson_r = corr_df.corr(method='pearson')
pearson_p = corr_pvalues(corr_df, 'pearson')
spearman_r = corr_df.corr(method='spearman')
spearman_p = corr_pvalues(corr_df, 'spearman')

fig_corr = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f'Pearson-korreláció (n = {len(corr_df)})',
        f'Spearman-rangkorreláció (n = {len(corr_df)})'
    ),
    horizontal_spacing=0.25
)

for i, (rho, pmat) in enumerate([(pearson_r, pearson_p), (spearman_r, spearman_p)], start=1):
    sub = px.imshow(rho, zmin=-1, zmax=1, color_continuous_scale='RdBu_r', text_auto=False)
    tr = sub.data[0]
    # A make_subplots-ba áthelyezett trace-hoz a coloraxis helyett fix colorscale-t és rögzített skálát állítunk
    if getattr(tr, 'coloraxis', None) and getattr(sub.layout, 'coloraxis', None) is not None:
        tr.colorscale = sub.layout.coloraxis.colorscale
        tr.coloraxis = None
    tr.zmin = -1
    tr.zmax = 1
    tr.text = star_cells(rho, pmat).values
    tr.texttemplate = '%{text}'
    tr.x = [corr_labels.get(c, c) for c in rho.columns]
    tr.y = [corr_labels.get(c, c) for c in rho.index]
    fig_corr.add_trace(tr, row=1, col=i)

fig_corr.update_layout(
    title_text=(
        f'Korrelációs hőtérkép (eladó minta, listwise n = {len(corr_df)}) — '
        '* p &lt; 0.05, ** p &lt; 0.01, *** p &lt; 0.001'
    ),
    template=PLOTLY_TEMPLATE,
    height=580
)
fig_corr.update_xaxes(tickangle=-30)
fig_corr.show()


In [ ]:
# Interaktív Többváltozós Eloszláselemzés (Plotly updatemenus)
eda_vars = [
    ('nm_ar_huf', '1. Fajlagos Négyzetméterár (HUF/m²)', 40),
    ('ar_millio_ft', '2. Kínálati Vételár (M Ft)', 30),
    ('alapterulet_nm', '3. Lakás Alapterület (m²)', 25),
    ('szobaszam_osszes', '4. Szobaszám (db)', 15)
]

fig_eda = go.Figure()
buttons = []

for i, (col, label, nbins) in enumerate(eda_vars):
    sub_fig = px.histogram(
        elado, x=col, color='varosresz', barmode='overlay', opacity=0.7,
        nbins=nbins, template=PLOTLY_TEMPLATE
    )
    for tr in sub_fig.data:
        tr.visible = (i == 0)
        fig_eda.add_trace(tr)

traces_per_var = len(elado['varosresz'].unique())

for i, (col, label, nbins) in enumerate(eda_vars):
    vis = [False] * len(fig_eda.data)
    for t_idx in range(i * traces_per_var, (i + 1) * traces_per_var):
        if t_idx < len(vis): vis[t_idx] = True
    buttons.append(dict(
        label=label,
        method='update',
        args=[{'visible': vis}, {'title': f'{label} eloszlása a(z) {AREA_NAME} városrészeiben', 'xaxis': {'title': label}}]
    ))

fig_eda.update_layout(
    title=f'{eda_vars[0][1]} eloszlása a(z) {AREA_NAME} városrészeiben',
    xaxis_title=eda_vars[0][1],
    yaxis_title='Gyakoriság (darabszám)',
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        direction='down',
        x=0.01, y=0.99, xanchor='left', yanchor='top',
        bgcolor='white', bordercolor='#cbd5e1'
    )],
    template=PLOTLY_TEMPLATE,
    height=480
)
fig_eda.show()